# Econometrics II — Assignment 1
## Forecasting Economic Growth and Growth-at-Risk

**Structure**
1. Setup & Data Loading
2. Graphical Analysis
3. Model Selection & Estimation
4. Misspecification Testing
5. Forecasting (2010Q1–2024Q4)
6. Growth-at-Risk (2010Q1–2024Q4)
7. Export Figures & Tables

## 1. Setup & Data Loading

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.stattools import jarque_bera
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

# Reproducibility / aesthetics
plt.rcParams.update({"figure.dpi": 150, "axes.spines.top": False, "axes.spines.right": False})
ALPHA = 0.10  # GaR risk level

In [ ]:
# Load data
df = pd.read_excel("Assignment_1.xlsx", index_col=0, parse_dates=True)
df.index = pd.PeriodIndex(df.index, freq="Q")
print(df.shape, df.columns.tolist())
df.head()

In [ ]:
# Define train / eval masks
TRAIN_END = "2009Q4"
EVAL_START = "2010Q1"
EVAL_END   = "2024Q4"

train = df.loc[:TRAIN_END, "y"]
eval_y = df.loc[EVAL_START:EVAL_END, "y"]
forecast_naive = df.loc[EVAL_START:EVAL_END, "forecast_naive"]
GaR_np = df.loc[EVAL_START:EVAL_END, "GaR_np"]

print(f"Training obs : {len(train)}")
print(f"Eval obs     : {len(eval_y)}")

## 2. Graphical Analysis

**Figure 1** — Full-sample time-series of $y_t$, evaluation period shaded.  
**Figure 2** — ACF and PACF of $y_t$ on training sample (lags 1–12).

In [ ]:
# Figure 1: Full-sample GDP growth with evaluation period shaded
fig, ax = plt.subplots(figsize=(10, 4))
# TODO: plot df["y"], shade eval period, add zero line
# ax.plot(...)
# ax.axvspan(...)
ax.set_title("Figure 1: U.S. Quarterly GDP Growth $y_t$")
ax.set_ylabel("Year-on-year growth")
plt.tight_layout()
# plt.savefig("figures/fig1_gdp_growth.pdf")
plt.show()

In [ ]:
# Figure 2: ACF and PACF on training sample
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
plot_acf(train, lags=12, ax=axes[0], title="ACF")
plot_pacf(train, lags=12, ax=axes[1], title="PACF")
fig.suptitle("Figure 2: ACF and PACF of $y_t$ (Training Sample 1990Q1–2009Q4)")
plt.tight_layout()
# plt.savefig("figures/fig2_acf_pacf.pdf")
plt.show()

## 3. Model Selection & Estimation

Estimate AR(1)–AR(4) via OLS and ARMA(1,1) via ML.  
Compare using AIC and BIC → **Table 1**.

In [ ]:
# Estimate AR(1)–AR(4) and ARMA(1,1) on training sample
# Store results in a dict keyed by model label
model_specs = {
    "AR(1)":     (1, 0, 0),
    "AR(2)":     (2, 0, 0),
    "AR(3)":     (3, 0, 0),
    "AR(4)":     (4, 0, 0),
    "ARMA(1,1)": (1, 0, 1),
}
fitted_models = {}
for label, (p, d, q) in model_specs.items():
    res = ARIMA(train, order=(p, d, q), trend="c").fit()
    fitted_models[label] = res

# Table 1: AIC, BIC, sigma-hat, Ljung-Box p-value (lag 8)
rows = []
for label, res in fitted_models.items():
    lb_pval = acorr_ljungbox(res.resid, lags=[8], return_df=True)["lb_pvalue"].values[0]
    rows.append({
        "Model": label,
        "AIC": round(res.aic, 2),
        "BIC": round(res.bic, 2),
        "σ̂": round(np.sqrt(res.params.get("sigma2", np.nan)), 4),
        "LB p-val (lag 8)": round(lb_pval, 3),
    })

table1 = pd.DataFrame(rows).set_index("Model")
print("Table 1: Model Comparison")
print(table1.to_string())

In [ ]:
# Select preferred model (update label after inspecting Table 1)
PREFERRED_LABEL = "AR(1)"  # <-- update after running Table 1
preferred = fitted_models[PREFERRED_LABEL]
print(preferred.summary())

## 4. Misspecification Testing

Run on preferred model residuals **before** interpreting coefficients.

| Test | H₀ | Lags / details |
|---|---|---|
| Ljung-Box | No serial autocorrelation | lags 4, 8 |
| Jarque-Bera | Residuals are normally distributed | — |
| ARCH-LM | No conditional heteroskedasticity | lags 4 |

In [ ]:
resid = preferred.resid.dropna()

# Ljung-Box at lags 4 and 8
lb = acorr_ljungbox(resid, lags=[4, 8], return_df=True)
print("Ljung-Box test:")
print(lb[["lb_stat", "lb_pvalue"]].rename(columns={"lb_stat": "LB stat", "lb_pvalue": "p-value"}))

# Jarque-Bera normality test
jb_stat, jb_pval, jb_skew, jb_kurt = jarque_bera(resid)
print(f"\nJarque-Bera: stat={jb_stat:.3f}, p-value={jb_pval:.3f}")

# ARCH-LM test (lags 4)
arch_stat, arch_pval, _, _ = het_arch(resid, nlags=4)
print(f"ARCH-LM (lag 4): stat={arch_stat:.3f}, p-value={arch_pval:.3f}")

In [ ]:
# Table 2: Preferred model estimates
params = preferred.params
bse   = preferred.bse
tstat = preferred.tvalues
pvals = preferred.pvalues

table2 = pd.DataFrame({
    "Coefficient": params,
    "Std. Error": bse,
    "t-stat": tstat,
    "p-value": pvals,
}).round(4)
print(f"Table 2: {PREFERRED_LABEL} Estimation Results (1990Q1–2009Q4)")
print(table2.to_string())

## 5. Forecasting (2010Q1–2024Q4)

Produce one-step-ahead forecasts using **fixed** training-sample parameters with recursive residual updates (per ARMA forecasting formula).  
Compare against `forecast_naive` via RMSE and Diebold-Mariano test → **Figure 3**, **Table 3**.

In [ ]:
# One-step-ahead forecasts using fixed training-sample parameters
# Apply the model to the full sample (train + eval) with parameters frozen from training
full_y = df.loc[:EVAL_END, "y"]

# Use apply() to obtain forecasts over eval period with fixed params
pred = preferred.apply(full_y)
mu_hat = pred.fittedvalues  # conditional mean mu_t

# Restrict to eval period
forecast_model = mu_hat.loc[EVAL_START:EVAL_END]

print(f"Forecast obs: {len(forecast_model)}")

In [ ]:
# Forecast evaluation: RMSE
e_model = eval_y.values - forecast_model.values
e_naive = eval_y.values - forecast_naive.values

rmse_model = np.sqrt(np.mean(e_model**2))
rmse_naive = np.sqrt(np.mean(e_naive**2))
print(f"RMSE model : {rmse_model:.4f}")
print(f"RMSE naive : {rmse_naive:.4f}")

# Diebold-Mariano test (H0: equal MSE)
d = e_model**2 - e_naive**2
T_eval = len(d)
dm_stat = np.mean(d) / (np.std(d, ddof=1) / np.sqrt(T_eval))
dm_pval = 2 * (1 - stats.norm.cdf(abs(dm_stat)))
print(f"\nDiebold-Mariano: DM={dm_stat:.3f}, p-value={dm_pval:.3f}")

In [ ]:
# Figure 3: Realised y_t vs model forecast vs naive forecast (eval period)
fig, ax = plt.subplots(figsize=(10, 4))
# TODO: plot eval_y, forecast_model, forecast_naive
ax.set_title("Figure 3: One-Step-Ahead Forecasts (2010Q1–2024Q4)")
ax.set_ylabel("Year-on-year growth")
ax.legend()
plt.tight_layout()
# plt.savefig("figures/fig3_forecasts.pdf")
plt.show()

## 6. Growth-at-Risk (2010Q1–2024Q4)

$$\widehat{\text{GaR}}^{0.1}_t = \hat{\mu}_t + \hat{\sigma} \cdot \Phi^{-1}(0.1)$$

Evaluate via violation rate, t-test for correct coverage (H₀: $E[\text{hit}_t]=0.1$), and tick loss → **Figure 4**, **Table 4**.

In [ ]:
# Compute GaR estimates
sigma_hat = np.sqrt(preferred.params.get("sigma2", preferred.params.iloc[-1]))
phi_inv   = stats.norm.ppf(ALPHA)  # Phi^{-1}(0.10) ≈ -1.282

GaR_model = forecast_model + sigma_hat * phi_inv

print(f"σ̂ = {sigma_hat:.4f}")
print(f"Φ⁻¹(0.10) = {phi_inv:.4f}")
print(f"\nModel GaR (first 5):\n{GaR_model.head()}")
print(f"GaR_np (first 5):\n{GaR_np.head()}")

In [ ]:
# Violation rate and coverage test
def evaluate_gar(y, gar, label="Model"):
    hit = (y.values < gar.values).astype(float)
    p_hat = hit.mean()
    T = len(hit)
    # t-test: H0: E[hit] = alpha
    se = np.sqrt(ALPHA * (1 - ALPHA) / T)
    t_stat = (p_hat - ALPHA) / se
    t_pval = 2 * (1 - stats.norm.cdf(abs(t_stat)))
    # Tick loss
    tick = ((ALPHA - hit) * (y.values - gar.values)).mean()
    print(f"--- {label} ---")
    print(f"Violation rate p̂ = {p_hat:.3f}  (target: {ALPHA})")
    print(f"t-stat = {t_stat:.3f},  p-value = {t_pval:.3f}")
    print(f"Mean tick loss = {tick:.6f}\n")
    return p_hat, t_stat, t_pval, tick

model_res = evaluate_gar(eval_y, GaR_model, label="AR Model GaR")
np_res    = evaluate_gar(eval_y, GaR_np,    label="Naive GaR_np")

In [ ]:
# Figure 4: Realised y_t vs model GaR vs GaR_np (eval period)
fig, ax = plt.subplots(figsize=(10, 4))
# TODO: plot eval_y, GaR_model, GaR_np
ax.set_title("Figure 4: Growth-at-Risk Estimates at 10% Level (2010Q1–2024Q4)")
ax.set_ylabel("Year-on-year growth")
ax.legend()
plt.tight_layout()
# plt.savefig("figures/fig4_gar.pdf")
plt.show()

## 7. Export Figures & Tables

Uncomment the `savefig` / `to_latex` calls in cells above once plots and tables are finalised.  
Place output in `figures/` and `tables/` subdirectories, then reference them in `assignment1.tex`.